### Adsorbate vibrations


Adsorbates have vibrational modes that can be computed using finite differences. For metal surfaces, it's common to freeze the metal atoms since they are much heavier than typical adsorbates.


#### O vibrations at fcc site


In [1]:
from vasp import Vasp
from ase.lattice.surface import fcc111, add_adsorbate
from ase.constraints import FixAtoms

# Create Pt slab with O at fcc site
atoms = fcc111('Pt', size=(2, 2, 3), vacuum=10.0)
add_adsorbate(atoms, 'O', height=1.2, position='fcc')

# Fix all Pt atoms - only O moves for vibrations
constraint = FixAtoms(mask=[atom.symbol != 'O' for atom in atoms])
atoms.set_constraint(constraint)

calc = Vasp(label='surfaces/Pt-O-fcc-vib',
            xc='PBE',
            encut=300,
            kpts=(3, 3, 1),
            ibrion=5,     # finite differences with selective dynamics
            nfree=2,      # central differences
            potim=0.015,
            ediff=1e-7,
            nsw=1,
            atoms=atoms)

energy = calc.get_potential_energy()
print(f'Energy: {energy:.4f} eV')

freqs, modes = calc.get_vibrational_modes()

# Convert to cm^-1
c = 3e10  # cm/s
h = 4.135667516e-15  # eV*s

print('\nVibrational frequencies:')
for i, f in enumerate(freqs):
    if isinstance(f, complex):
        print(f'  Mode {i}: {f.real/(h*c):.1f} cm^-1 (imaginary)')
    else:
        print(f'  Mode {i}: {f/(h*c):.1f} cm^-1')

/opt/conda/lib/python3.9/site-packages/ase/lattice/surface.py:17: UserWarning: Moved to ase.build
  warnings.warn('Moved to ase.build')


Energy: -74.8334 eV

Vibrational frequencies:
  Mode 0: 4356079.6 cm^-1
  Mode 1: 3557931.2 cm^-1
  Mode 2: 3556591.0 cm^-1


The highest frequency mode corresponds to the O-surface stretch (perpendicular to surface). The lower frequency modes are frustrated translations parallel to the surface.


#### O vibrations at bridge site (transition state)


In [2]:
# O at bridge site - this is a saddle point
atoms_br = fcc111('Pt', size=(2, 2, 3), vacuum=10.0)
add_adsorbate(atoms_br, 'O', height=1.2, position='bridge')
constraint = FixAtoms(mask=[atom.symbol != 'O' for atom in atoms_br])
atoms_br.set_constraint(constraint)

calc_br = Vasp(label='surfaces/Pt-O-bridge-vib',
               xc='PBE',
               encut=300,
               kpts=(3, 3, 1),
               ibrion=5,
               nfree=2,
               potim=0.015,
               ediff=1e-7,
               nsw=1,
               atoms=atoms_br)

energy_br = calc_br.get_potential_energy()
print(f'Energy: {energy_br:.4f} eV')

freqs_br, modes_br = calc_br.get_vibrational_modes()

print('\nVibrational frequencies:')
has_imaginary = False
for i, f in enumerate(freqs_br):
    if isinstance(f, complex):
        print(f'  Mode {i}: {f.real/(h*c):.1f} cm^-1 (IMAGINARY)')
        has_imaginary = True
    else:
        print(f'  Mode {i}: {f/(h*c):.1f} cm^-1')

if has_imaginary:
    print('\nOne imaginary mode indicates this is a transition state!')
    print('The bridge site is a saddle point for diffusion between fcc and hcp sites.')

Energy: -73.6507 eV

Vibrational frequencies:
  Mode 0: 5502426.4 cm^-1
  Mode 1: 5492265.4 cm^-1
  Mode 2: -2496139.2 cm^-1
